# ⭐ Biophysical Constraints

Understanding the physical laws that govern protein geometry in ResonanceFlow: steric clashes and virtual Cα-Cα bond lengths.

We use differentiable penalty functions (losses) to ensure the 3D structures we generate obey basic chemistry.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/resonance-flow/blob/main/examples/interactive_tutorials/biophysical_constraints.ipynb)

In [ ]:
# Setup: Install ResonanceFlow and Matplotlib
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q resonance-flow matplotlib numpy
else:
    import os

    sys.path.insert(0, os.path.abspath("../../"))

## 1. Virtual Bond Length (Cα–Cα)

In coarse-grained models, we only track the alpha-carbon (Cα) atoms. The distance between consecutive Cα atoms is fixed by the rigid planar peptide bond geometry to be approximately **3.80 Å**.

ResonanceFlow penalises deviations from 3.8 Å using a harmonic potential: $V(r) = (r - 3.8)^2$.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from resonance_flow import get_bond_length_loss

bond_fn = get_bond_length_loss(target_distance=3.8)

# Let's test a chain with 3 atoms (2 bonds) at ideal spacing
ca_chain_ideal = jnp.array([[0.0, 0.0, 0.0], [3.8, 0.0, 0.0], [7.6, 0.0, 0.0]])
print(f"Bond Loss (ideal 3.8 Å): {bond_fn(ca_chain_ideal):.4f}")

# Let's test a compressed chain (3.0 Å spacing)
ca_chain_compressed = jnp.array([[0.0, 0.0, 0.0], [3.0, 0.0, 0.0], [6.0, 0.0, 0.0]])
print(f"Bond Loss (compressed 3.0 Å): {bond_fn(ca_chain_compressed):.4f}")

Let's visualize the shape of this harmonic penalty well.

In [ ]:
r = np.linspace(2.0, 5.5, 100)
losses = [(r_val - 3.8) ** 2 for r_val in r]

plt.figure(figsize=(7, 4))
plt.plot(r, losses, "b-", lw=2)
plt.axvline(3.8, color="k", linestyle="--", alpha=0.5, label="Ideal (3.8 Å)")
plt.title("Harmonic Cα-Cα Virtual Bond Penalty")
plt.xlabel("Cα-Cα Distance (Å)")
plt.ylabel("Loss (Energy)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. Steric Clash Repulsion

Two atoms cannot occupy the same physical space due to Pauli repulsion. ResonanceFlow models this using a truncated harmonic "flat-bottom" potential. If the distance between two atoms is *greater* than the sum of their van der Waals radii, the loss is zero. If they overlap, the loss increases quadratically.

$V(r) = \max(0, R_{vdw1} + R_{vdw2} - r)^2 / 2$

In [ ]:
from resonance_flow.losses import get_steric_clash_loss

steric_fn = get_steric_clash_loss(exclude_bonded_range=0)

radius = 1.7  # approximate vdW radius for Carbon in Å
radii_sum = radius + radius

r_steric = np.linspace(1.0, 5.0, 100)
steric_losses = [0.5 * max(0, radii_sum - r_val) ** 2 for r_val in r_steric]

plt.figure(figsize=(7, 4))
plt.plot(r_steric, steric_losses, "r-", lw=2)
plt.axvline(
    radii_sum, color="k", linestyle="--", alpha=0.5, label=f"Contact Distance ({radii_sum:.1f} Å)"
)
plt.axvspan(1.0, radii_sum, color="red", alpha=0.1, label="Clash Region")
plt.title("Steric Repulsion Penalty (Truncated Harmonic)")
plt.xlabel("Distance between atoms (Å)")
plt.ylabel("Loss (Energy)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()